# Candidate events, case III

This R notebook generates list of candidate events - poison exons followed by a PAS. Specifically, annotated alternative terminal exons linked to the downstream part of the gene by a splice junction from GTEx data.

Short workflow:
  - load annotation, identify alternative terminal exons (ATE)
  - load preprocessed unannotated splice junctions from GTEx
  - intersect the ATE with junctions
    - (1) the acceptor splice site of the splice junction coincides with the end of the intron that contains the ATE, 
    - (2) the donor splice site of the splice junction is located >50 nts downstream of the annotated stop codon and either within the exon or <550 nts downstream of the stop codon (the threshold selection process is included)
  - for the selected list of events generate tables with genomic coordinates of all 4 splice sites, corresponding junctions and constitutive exons surrounding the ATE.
    - output, saved to `ATEPE_from_novel_junc_data/` folder: `novel_junc_ATEPE.n260.skip.tsv`, `ATE_PE.junctions.PE_id.n260.tsv`,`ATE_PE_ss1_ss4.PE_id.n260.tsv`,`ATE_PE_e1_e2.PE_id.uniq_sur_ex.n260.bed`, `ATE_PE_e1_e2.PE_id.n260.bed`.

In [10]:
library(dplyr)
library(tidyr)
library(stringr)
library(rtracklayer)

library(data.table)
library(purrr)


Attaching package: ‘dplyr’


The following object is masked from ‘package:AnnotationDbi’:

    select


The following object is masked from ‘package:Biobase’:

    combine


The following objects are masked from ‘package:GenomicRanges’:

    intersect, setdiff, union


The following object is masked from ‘package:GenomeInfoDb’:

    intersect


The following objects are masked from ‘package:IRanges’:

    collapse, desc, intersect, setdiff, slice, union


The following objects are masked from ‘package:S4Vectors’:

    first, intersect, rename, setdiff, setequal, union


The following objects are masked from ‘package:BiocGenerics’:

    combine, intersect, setdiff, union


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘tidyr’


The following object is masked from ‘package:S4Vectors’:

    expand




In [2]:
ref_chr <- paste0("chr",c(1:22,"Y","X"))
data_dir = "../"

## Annotation from TxDb

In [26]:
library(GenomicFeatures)
library(AnnotationFilter)

load transcript annotation

In [ ]:
chess_txdb <- loadDb(paste0(data_dir,"references/chess3.1.3.GRCh38.txdb.sqlite"))

load txt with transcript ids of protein coding genes (to transcript_type tag in sqlite format)

In [ ]:
prot_cod <- read.table(paste0(data_dir,"references/chess3.1.3.GRCh38.prot_cod_trid.txt"),stringsAsFactors = F)[,1]
length(prot_cod)

[1] 105328

Extract exons of the protein coding transcripts in prot_cod from txdb

In [ ]:
ex <- select(chess_txdb, keys = prot_cod, keytype = "TXNAME",
        columns=c('EXONCHROM','EXONSTART','EXONEND','EXONSTRAND','TXNAME','GENEID'))#'EXONRANK',
colnames(ex)  <- c("transcript_id","seqid","strand","start","end","gene_id")#,,"ex_rank"

ex <- ex[ex$seqid %in% ref_chr,]
dim(ex)

ex <- arrange(ex, transcript_id, start) %>% 
    mutate(ss4 = ifelse(transcript_id == c(transcript_id[-1],"-"), c(start[-1],"-"),"-"),
            ss1 = ifelse(transcript_id == c("-", transcript_id[-length(transcript_id)]), c("-", end[-length(end)]), "-"))
dim(ex)

'select()' returned 1:many mapping between keys and columns



[1] 1197828       6

[1] 1197828       8

merge ovrlapping genes

In [ ]:
gene_gr <- genes(chess_txdb, columns = "gene_id", filter = list(tx_name = c(prot_cod)))

In [ ]:
mgenes_gr <- reduce(gene_gr, with.revmap = T)
mgenes_gr$gene_id <- extractList(mcols(gene_gr)$gene_id, mgenes_gr$revmap)

mgenes_id <- unlist(
    lapply(seq_along(mgenes_gr$revmap), function(i) rep(i,length(mgenes_gr$revmap[[i]]))))
names(mgenes_id) <- unlist(mgenes_gr$gene_id)
ex$mgene <- paste0("g",mgenes_id[ex$gene_id])

rm(gene_gr, mgenes_gr)

In [ ]:
#save exon list to RData
#save(ex, file = "ex.cassette_ex.ATEPE-fromjunc.RData")

### cassette exons

identify cassette exons among all exons

In [ ]:
pure_cas <- ex %>% 
    dplyr::rename(ss2 = start, ss3 = end) %>% 
    mutate(ex = paste(seqid, ss2, ss3, strand, sep = "_"),
        I1 = paste(ss1, ss2, sep = "_"),
        I2 = paste(ss3, ss4, sep = "_"),
        E = paste(ss1, ss4, sep = "_"), .keep = "unused") %>%    
    dplyr::select(-c(gene_id, transcript_id)) %>%  
    unique %>%   
    group_by(mgene) %>% 
    mutate(cas1 = apply(outer(E, I1, FUN = "=="), 1, any),
        cas2 = apply(outer(E, I2, FUN = "=="), 1, any), 
        other1 = apply(outer(I1, I1, FUN = "=="), 1, sum) - 1,
        other2 = apply(outer(I2, I2, FUN = "=="), 1, sum) - 1) 

cas <- pure_cas %>% 
    filter(cas1 & cas2) %>% 
    dplyr::select(-starts_with(c("other", "cas"))) %>%
    ungroup

Get pure cassette exons (does not share I1 or I2 junction with other exons) that are allowed to share junction with ATE.
ATE - alternative terminal exon

In [ ]:
pure_cas_nonterminal0 <- ex %>%   
    dplyr::rename(ss2 = start, ss3 = end) %>% 
    mutate(
        ex = paste(seqid, ss2, ss3, strand, sep = "_"),
        I1 = paste(ss1, ss2, sep = "_"),
        I2 = paste(ss3, ss4, sep = "_"),
        E = paste(ss1, ss4, sep = "_")) %>%  
    filter(!(ss1 == "-") & !(ss4 == "-")) %>% 
    dplyr::select(-starts_with("ss"), -seqid) %>% 
    dplyr::select(-c(gene_id, transcript_id, strand)) %>%  
    unique %>% 
    group_by(mgene) %>% 
    mutate(
        other1_nt = apply(outer(I1, I1, FUN="=="), 1, sum) - 1,
        other2_nt = apply(outer(I2, I2, FUN="=="), 1, sum) - 1) %>% 
    left_join(cas,.)

pure_cas_nonterminal <- pure_cas_nonterminal0 %>% 
    filter(other1_nt == 0 & other2_nt == 0) %>% 
    dplyr::select(-starts_with("other")) %>% 
    ungroup 

## Novel PE from GTEx junctions

### get skipped ATE in introns

In [ ]:
cte3 <- ex %>%   
    dplyr::rename(ss2 = start, ss3 = end) %>% 
    mutate(
        ex = paste(seqid, ss2, ss3, strand, sep = "_"),
        I1 = paste(ss1, ss2, sep = "_"),
        I2 = paste(ss3, ss4, sep = "_"),
        E = paste(ss1, ss4, sep = "_")) %>%    
    dplyr::select(-c(gene_id, transcript_id)) %>%  
    unique %>%   
    group_by(mgene) %>% 
    mutate(
        in_intron = apply(
            (strand == "+" & outer(ss1, ss1, FUN = "==") &
            outer(ss2, ss1, FUN = ">") & 
            outer(ss3, ss2, FUN = "<")) |
            (strand == "+" & 
            outer(ss1, ss3, FUN = "==") & 
            outer(ss2, ss3, FUN = ">") & 
            outer(ss3, ss4, FUN = "<") & 
            !outer(ss4, ss4, FUN = "==")) |
            (strand == "-" & 
            outer(ss4, ss2, FUN = "==") & 
            outer(ss2, ss1, FUN = ">") & 
            outer(ss3, ss2, FUN = "<") & 
            !outer(ss1, ss1, FUN = "==")) |
            (strand == "-" & 
            outer(ss4, ss4, FUN = "==") & 
            outer(ss2, ss3, FUN = ">") & 
            outer(ss3, ss4, FUN = "<")),
            1, sum),
        #outer(ex within E & ss1==ss1)
        other_I1 = apply(outer(I1, I1, FUN = "=="), 1, sum) - 1,
        other_I2 = apply(outer(I2, I2, FUN = "=="), 1, sum) - 1) %>% 
    ungroup %>% 
    filter(in_intron > 0 & ((ss4 == "-" & strand == "+") | (ss1 == "-" & strand == "-"))) %>% 
    filter(!(ss4 == "-" & ss1 == "-"))

In [18]:
n_distinct(cte3$ex)

[1] 4188

among them select ATE that do not share I1 with any exon or share it with other terminal exons

In [20]:
cte_wterm <- ex %>%   
    dplyr::rename(ss2 = start, ss3 = end) %>% 
    mutate(ex = paste(seqid, ss2, ss3, strand, sep = "_"),
           I1 = paste(ss1, ss2, sep = "_"),
           I2 = paste(ss3, ss4, sep = "_"),
           E = paste(ss1, ss4, sep = "_")) %>%    
    filter((ss1 == "-") | (ss4 == "-")) %>% 
    dplyr::select(-starts_with("ss"), -seqid) %>% 
    dplyr::select(-c(gene_id, transcript_id)) %>%  
    unique %>%   
    group_by(mgene) %>% 
    mutate(othert_I1 = apply(outer(I1, I1, FUN = "=="), 1, sum) - 1,
           othert_I2 = apply(outer(I2, I2, FUN = "=="), 1, sum) - 1) %>% 
    left_join(cte3, .)   %>% 
    dplyr::select(-starts_with("ss"))

Joining with `by = join_by(strand, mgene, ex, I1, I2, E)`


Add trasncript ids

In [21]:
cte_wterm <- ex %>%   
    dplyr::rename(ss2 = start, ss3 = end) %>% 
    mutate(ex = paste(seqid, ss2, ss3, strand, sep = "_"),
           I1 = paste(ss1, ss2, sep = "_"),
           I2 = paste(ss3, ss4, sep = "_"),
           E = paste(ss1, ss4, sep = "_")) %>% 
    filter(ex %in% cte_wterm$ex) %>%   
    group_by(ex, I1, I2, E) %>% 
    summarize(transcript_id = paste(transcript_id, collapse = ","), .groups="drop") %>%  
    left_join(cte_wterm, .)

Joining with `by = join_by(ex, I1, I2, E)`


Stop codons

get coordinates of stop codons of our ATEs from TxDb

In [ ]:
tr_id_ofcte <- unique(unlist(str_split(cte_wterm$transcript_id, pattern = ",")))
stops <- AnnotationDbi::select(chess_txdb, keys = tr_id_ofcte, keytype = "TXNAME", 
        columns = c('CDSEND', 'CDSSTART', 'CDSSTRAND', 'TXNAME')) %>% 
    group_by(TXNAME) %>% 
    summarize(stop = ifelse(CDSSTRAND[1] == "+", max(CDSEND), min(CDSSTART)))
stops <- setNames(stops$stop, stops$TXNAME)

In [28]:
cte_wstop <- cte_wterm %>% 
    data.frame %>% 
    mutate(tr_id = str_split(transcript_id, pattern = ","), .keep = "unused") %>% 
    unnest(tr_id) %>% 
    mutate(stop = stops[tr_id]) %>% 
    filter(stop > str_split_fixed(ex, pattern = "_", n = 4)[,2] & 
           stop < str_split_fixed(ex, pattern = "_", n = 4)[,3]) %>%  
    group_by(mgene, ex, I1, I2, E, stop) %>% 
    summarize(across(starts_with("other"), ~ .x[1]),
              tr_id = paste(tr_id, collapse = ","),
              .groups = "drop")

In [31]:
#save(cte_wstop, file = "cte_wstop.cassette-ex.ATEPE_fromjunc.RData")

I2 for ATE (finding the ends of introns that exclude ATE)

In [ ]:
cte_wstop_ss4 <- ex %>% 
    filter(mgene %in% cte_wstop$mgene) %>% 
    dplyr::rename(ss2 = start, ss3 = end) %>% 
    mutate(ex = paste(seqid, ss2, ss3, strand, sep = "_")) %>% 
    dplyr::select(-c(seqid, gene_id, transcript_id)) %>% 
    unique %>% 
    inner_join(., 
        dplyr::select(cte_wstop, -c(I1, I2, tr_id, stop, starts_with("other"))),
        by = join_by(mgene), suffix = c(".ex",".ate")) %>% 
    dplyr::rename(E.ate=E) %>% 

mutate(
    ss1.ate = str_split_fixed(E.ate, pat = "_", n = 2)[, 1],
    ss2.ate = str_split_fixed(ex.ate, pat = "_", 4)[, 2],
    ss3.ate = str_split_fixed(ex.ate, pat = "_", 4)[,3],
    ss4.ate = str_split_fixed(E.ate, pat = "_", 2)[, 2],
    across(starts_with("ss"), as.integer)) %>% 
    group_by(mgene, ex.ate, E.ate) %>% 
    filter(
        (strand == "+" & ss1.ate == ss1 & ss3.ate < ss2) |
        (strand == "+" & ss1.ate == ss3 & ss3.ate < ss4 & !(is.na(ss4))) |
        (strand == "-" & ss4.ate == ss4 & ss2.ate > ss3) | 
        (strand == "-" & ss4.ate == ss2 & ss2.ate > ss1 & !(is.na(ss1))) ) %>% 
    ungroup %>% 

    mutate(
        ss4.ate = case_when(
            strand == "+" & ss1.ate == ss1 ~ ss2,
            strand == "+" & ss1.ate == ss3 ~ ss4,
            .default = ss4.ate),
        ss1.ate = case_when(
            strand == "-" & ss4.ate == ss4 ~ ss3,
            strand == "-" & ss4.ate == ss2 ~ ss1,
            .default = ss1.ate)) %>%   
    dplyr::select(-c(ss1, ss2, ss3, ss4, ex.ex, strand, ss2.ate, ss3.ate, mgene)) %>% 
    unique %>% #arrange(ex.ate,E.ate) 
    inner_join(cte_wstop,., by = join_by(ex == ex.ate, E == E.ate))

#leave only the shortest intron
cte_wstop_ss4 <- cte_wstop_ss4 %>% 
    group_by(ex, I1, I2, E, stop) %>% 
    filter(
        (grepl(ex, pat = "+", fixed = T) & ss4.ate == min(ss4.ate, na.rm = T)) | 
        (grepl(ex, pat = "-", fixed = T) & ss1.ate == max(ss1.ate, na.rm = T))) %>% 
    ungroup %>%   
    filter( other_I1 == othert_I1 & other_I2 == othert_I2) %>% 
    dplyr::select(-starts_with("other"))

Warning message in inner_join(., dplyr::select(cte_wstop, -c(I1, I2, tr_id, stop, :
“Detected an unexpected many-to-many relationship between `x` and `y`.
ℹ Row 1 of `x` matches multiple rows in `y`.
ℹ Row 79 of `y` matches multiple rows in `x`.
ℹ If a many-to-many relationship is expected, set `relationship = "many-to-many"` to silence this warning.”
Warning message:
“There were 4 warnings in `mutate()`.
The first warning was:
ℹ In argument: `across(starts_with("ss"), as.integer)`.
Caused by warning:
! NAs introduced by coercion
ℹ Run `dplyr::last_dplyr_warnings()` to see the 3 remaining warnings.”
Warning message in inner_join(cte_wstop, ., by = join_by(ex == ex.ate, E == E.ate)):
“Detected an unexpected many-to-many relationship between `x` and `y`.
ℹ Row 3 of `x` matches multiple rows in `y`.
ℹ Row 421 of `y` matches multiple rows in `x`.
ℹ If a many-to-many relationship is expected, set `relationship = "many-to-many"` to silence this warning.”


In [78]:
n_distinct(cte_wstop_ss4$ex)

[1] 4188

[1] 3799

[1] 3201

### Intersect [stop_codon, ss4] with GTEx junctions

get junctions from gtex

In [ ]:
junc <- fread(file = paste0(data_dir, "GTEX_SJ/GTEX_junctions_sum_min10.hg38.tab"))
junc <- junc %>% filter(V3 < 3 & V3 > 0) #only unannotated junc with an annotated ss (IPSA)

In [ ]:
junc_gr <- data.frame(str_split_fixed(junc$V1, n = 4, pattern = "_"), reads = junc$V2) %>%   
    makeGRangesFromDataFrame(
        seqnames.field = "X1",
        start.field = "X2",
        end.field = "X3",
        strand.field = "X4",
        keep.extra.columns = T)

ATE to GRanges fro intersection with junctions

In [ ]:
cte_stop_gr <- cte_wstop_ss4 %>% 
    dplyr::rename(stop_cod = stop) %>%  
    mutate(chr = str_split_fixed(ex, pat = "_",4)[,1],
        strand = str_split_fixed(ex, pat = "_",4)[,4],
        start = ifelse(strand == "+", stop_cod, ss1.ate),
        end = ifelse(strand == "+", ss4.ate, stop_cod)) %>% 
    dplyr::select(-starts_with("ss"), -c(mgene, I1, I2)) %>%  
    makeGRangesFromDataFrame(keep.extra.columns = T)
cte_stop_gr <- sort(cte_stop_gr)
length(cte_stop_gr)

[1] 3283

intersection (taking into account the 50bp rule)

In [ ]:
table(width(cte_stop_gr)>50)

cte_stop50_gr <- c(
    narrow(cte_stop_gr[strand(cte_stop_gr) == "+" & width(cte_stop_gr) > 50], start = 50),
    narrow(cte_stop_gr[strand(cte_stop_gr) == "-" & width(cte_stop_gr) > 50], end = -50))

cte_stop50_gr$junc_sameend <- (overlapsAny(cte_stop50_gr, junc_gr, maxgap = 0L, type = "end") & as.vector(strand(cte_stop50_gr) == "+")) | 
    (overlapsAny(cte_stop50_gr, junc_gr, maxgap = 0L, type = "start") & as.vector(strand(cte_stop50_gr) == "-"))
cte_stop50_gr$junc_withinE <- FALSE
cte_stop50_gr$junc_withinE[unique(subjectHits(findOverlaps(junc_gr, cte_stop50_gr, type = "within")))] <- TRUE
cte_stop50_gr <- sort(cte_stop50_gr)


FALSE  TRUE 
    4  3279 

In [ ]:
cte_stop50_junc <- with(
    mergeByOverlaps(junc_gr, cte_stop50_gr, type = "within"), 
    cbind(
        as.data.frame(cte_stop50_gr)[c("ex", "E", "stop_cod", "start", "end", "tr_id")], 
        as.data.frame(junc_gr)[c("start", "end", "reads")]))
colnames(cte_stop50_junc) <- c('ex', 'E', 'stop_cod', 'ss1.ate', 'ss4.ate', 'tr_id', 'ss1.junc', 'ss4.junc', 'reads')
cte_stop50_junc <- cte_stop50_junc %>% 
    filter(
        (grepl(x = ex, pat = "+", fixed = T) & ss4.ate == ss4.junc) | 
        (grepl(x = ex, pat = "-", fixed = T) & ss1.ate == ss1.junc))

[1] 1290

[1] 1231

Filtering:

* only the closest intron end. 
* often another TSS is annotated in the intron in CHESS. so I deleted the ATE if there is a TSS between the stop_codon and ss4. (ex: "chr9_113541320_113549768_+", "chr9_95051111_95052134_+", "chr4_185312544_185316074_+")
* restricted distance between stop codon end and junction end (threshold from annotated poison exons)

examples: "chr2_201136198_201139878_+", "chr3_133935906_133936326_-", "chr17_46527088_46528859_-" annotated in GTEx

**delete ATE within introns with TSS**
get TSS:

In [ ]:
tr_gr <- transcripts(chess_txdb, columns = "tx_name", filter = list(tx_name = c(prot_cod)))

filter ATE:

In [ ]:
n_distinct(cte_stop50_gr$ex)
cte_stop50_noTSS_gr <- cte_stop50_gr[!overlapsAny(cte_stop50_gr, resize(tr_gr, width = 1, fix = "start"))]
n_distinct(cte_stop50_noTSS_gr$ex)

[1] 3197

[1] 3024

intersect junctions with updated ATE list

In [ ]:
cte_stop50_junc_noTSS <- with(
    mergeByOverlaps(junc_gr, cte_stop50_noTSS_gr, type = "within"), 
    cbind(
        as.data.frame(cte_stop50_noTSS_gr)[c("ex", "E", "stop_cod", "start", "end", "tr_id")], 
        as.data.frame(junc_gr)[c("start", "end", "reads")]))
colnames(cte_stop50_junc_noTSS) <- c('ex','E','stop_cod','ss1.ate','ss4.ate','tr_id','ss1.junc','ss4.junc','reads')
n_distinct(cte_stop50_junc_noTSS$ex)
cte_stop50_junc_noTSS <- cte_stop50_junc_noTSS %>% 
    filter(
        (grepl(x = ex, pat = "+", fixed = T) & ss4.ate == ss4.junc) | 
        (grepl(x = ex, pat = "-", fixed = T) & ss1.ate == ss1.junc))
n_distinct(cte_stop50_junc_noTSS$ex)

[1] 1145

[1] 1092

How to filter novel PE by 3UTR size:
 based on poison exon 3UTR length distribution (exon end - stop codon)

**poison exons for UTR length threshold**

get poison exons and their stop codon coords

In [ ]:
stops <- select(chess_txdb, keys = prot_cod, keytype = "TXNAME", 
    columns = c('CDSCHROM','CDSEND','CDSSTART','CDSSTRAND','CDSPHASE','TXNAME')) %>% 
    filter(!is.na(CDSCHROM)) %>% 
    group_by(TXNAME) %>%   
    summarize(chr = CDSCHROM[1], strand = CDSSTRAND[1],
            stop = ifelse(strand == "+", max(CDSEND, na.rm = T), min(CDSSTART, na.rm = T)))
dim(stops)
#stops <- setNames(stops$stop,stops$TXNAME)

'select()' returned 1:many mapping between keys and columns



[1] 105286      4

In [ ]:
stop_gr <- with(stops, 
    GRanges(seqnames = chr, strand = strand, ranges = IRanges(start = stop, width = 1), transcript_id_stop = TXNAME))

cas_gr <- ex %>% 
    mutate(
        ex = paste(seqid, start, end, strand, sep="_"),
        E = paste(ss1, ss4, sep="_")) %>% 
    dplyr::select(ex, seqid, start, end, strand, transcript_id, E) %>% 
    left_join(pure_cas_nonterminal,.,by = c("ex" = "ex", "E" = "E")) %>% 
    with(., GRanges( 
            seqnames = seqid, strand=strand, ranges = IRanges(start = start, end = end),
            ex = ex, E = E, I1 = I1, I2 = I2, transcript_id = transcript_id))

In [ ]:
poison_ex <- with(
    mergeByOverlaps(cas_gr, stop_gr), 
    cbind(
        as.data.frame(cas_gr)[c("ex","transcript_id","E","I1","I2")], 
        as.data.frame(stop_gr)[c("start","transcript_id_stop")])) %>% 
    filter(transcript_id_stop == transcript_id) %>% 
    dplyr::select(-"transcript_id_stop") %>% #unique %>% 
    dplyr::rename(stop_cod = start)

n_distinct(poison_ex$ex)

[1] 1388

In [ ]:
poison_ex %>% 
    mutate(
        utr = abs(stop_cod - as.integer(
            ifelse(
                grepl(x = ex, "+", fixed = T),
                str_split_fixed(ex, pat = "_", 4)[, 3],
                str_split_fixed(ex, pat = "_", 4)[, 2])))) %>% 
    filter(utr > 50) %>% 
    with(., quantile(utr, 0.75) + 1.5 * IQR(utr))
#ggplot(aes(x = utr)) +
#geom_boxplot(outlier.shape = NA) +
#coord_cartesian(xlim = c(0,500))

40%    45%    50%    55%    60%    65%    70%    75%    80%    85%    90% 
  91.0   97.0  103.0  111.0  120.0  130.0  141.9  155.0  170.0  197.9  249.9 
   95%   100% 
 552.6 2657.0

ERROR: Error in with(., quantile(utr, 0.75) + 1.5 * IQR(utr)): object '.' not found


number of CTE-PE depending on read threshold:
UTR length up to 250nt (90% of PE have shorter UTRs), 273.5 (UQ+1.5* IQR) or 550 (less than 95% utr) => The threshold of 550nts was selected as the 95th percentile of the 3’UTR length distribution for annotated poison104
exons.

### Final list and files with I1, I2, e1, e2  coordinates

In [ ]:
cte_junc_filt <- cte_stop50_junc_noTSS %>% 
    filter(reads >= 100) %>% #& abs(junc_dss-stop_cod)<2000) %>%
    mutate(
        within_ex = ifelse(
            grepl(x = ex, pat = "+", fixed = T),
            ss1.junc < as.integer(str_split_fixed(ex,pat="_",4)[,3]),
            as.integer(str_split_fixed(ex,pat="_",4)[,2]) < ss4.junc),
        UTR_len = ifelse(
            grepl(x = ex, pat = "+", fixed = T),
            ss1.junc - stop_cod, 
            stop_cod - ss4.junc)) %>% 
    filter(UTR_len > 0 & (UTR_len <= 550 | within_ex)) %>% 
    arrange(desc(reads)) %>% unique 

In [ ]:
#save the final list to table
write.table(cte_junc_filt, file = "novel_junc_ATEPE.n260.skip.tsv", quote = F, row.names = F, col.names = T, sep = "\t")
n_distinct(cte_junc_filt$ex)

[1] 260

In [ ]:
#table with 
cte_junc_filt %>% 
    mutate(chr=str_split_fixed(ex,pat="_",n=4)[,1],
        strand=str_split_fixed(ex,pat="_",n=4)[,4],
        ss1=ifelse(strand=="+",str_split_fixed(E,pat="_",n=2)[,1],ss1.junc),
        ss4=ifelse(strand=="+",ss4.junc,str_split_fixed(E,pat="_",n=2)[,2]),
        ss2=ifelse(strand=="+",str_split_fixed(ex,pat="_",n=4)[,2],ss4.junc),
        ss3=ifelse(strand=="+",ss1.junc,str_split_fixed(ex,pat="_",n=4)[,3])) %>% 
    mutate(
        I1=paste(chr,ss1,ss2,strand,sep="_"),
        I2=paste(chr,ss3,ss4,strand,sep="_"),
        E=paste(chr,ss1,ss4,strand,sep="_"),
        ex=paste(chr,ss2,ss3,strand,ss1,ss4,sep="_"),.keep="unused") %>% 
    #pull(ex) %>% n_distinct()
    write.table(.,file = "ATEPE_from_novel_junc_data/novel_junc_ATEPE.PE_id.n260.tsv", 
    quote = F, row.names = F, col.names = T, sep = "\t")

with junctions and all ss coordinates

In [ ]:
cte_junc_filt %>% 
dplyr::select(-c(stop_cod,tr_id,reads,within_ex,UTR_len)) %>% unique %>% 

mutate(chr=str_split_fixed(ex,pat="_",n=4)[,1],strand=str_split_fixed(ex,pat="_",n=4)[,4],
       ss1=ifelse(strand=="+",str_split_fixed(E,pat="_",n=2)[,1],ss1.junc),
       ss4=ifelse(strand=="+",ss4.junc,str_split_fixed(E,pat="_",n=2)[,2]),
       ss2=ifelse(strand=="+",str_split_fixed(ex,pat="_",n=4)[,2],ss4.junc),
       ss3=ifelse(strand=="+",ss1.junc,str_split_fixed(ex,pat="_",n=4)[,3])) %>% 
mutate(
        I1=paste(chr,ss1,ss2,strand,sep="_"),
        I2=paste(chr,ss3,ss4,strand,sep="_"),
        E=paste(chr,ss1,ss4,strand,sep="_"),
        ex=paste(chr,ss2,ss3,strand,ss1,ss4,sep="_")) %>% 
dplyr::select(-c(starts_with("ss"),chr,strand)) %>% unique %>%    
pivot_longer(cols = c(E,I1,I2)) %>%   
write.table(.,file = "ATEPE_from_novel_junc_data/ATE_PE.junctions.PE_id.n260.tsv",
         quote = F,row.names = F,col.names=F, sep="\t")

In [ ]:
cte_junc_filt %>% 
dplyr::select(-c(stop_cod,tr_id,reads,within_ex,UTR_len,ss1.ate,ss4.ate)) %>% unique %>% 

mutate(chr=str_split_fixed(ex,pat="_",n=4)[,1],strand=str_split_fixed(ex,pat="_",n=4)[,4],
       ss1=ifelse(strand=="+",str_split_fixed(E,pat="_",n=2)[,1],ss1.junc),
       ss4=ifelse(strand=="+",ss4.junc,str_split_fixed(E,pat="_",n=2)[,2]),
       ss2=ifelse(strand=="+",str_split_fixed(ex,pat="_",n=4)[,2],ss4.junc),
       ss3=ifelse(strand=="+",ss1.junc,str_split_fixed(ex,pat="_",n=4)[,3])) %>% 
mutate(ex=paste(chr,ss2,ss3,strand,ss1,ss4,sep="_"),
       ss1=paste(chr,ss1,strand,sep="_"),ss4=paste(chr,ss4,strand,sep="_")) %>% 
dplyr::select(-c(chr,strand,E,ss2,ss3,ss1.junc,ss4.junc)) %>% unique  %>%   
pivot_longer(cols = c(ss1,ss4)) %>% 
arrange(value) %>% 
write.table(.,file = "ATEPE_from_novel_junc_data/ATE_PE_ss1_ss4.PE_id.n260.tsv",
         quote = F,row.names = F,col.names=F, sep="\t")

Surrounding exons coordinates (e1, e2)

In [112]:
cte_junc_surex <- cte_junc_filt %>% 
dplyr::select(-c(tr_id,stop_cod,reads,within_ex,UTR_len,ss1.ate,ss4.ate)) %>% unique %>% 
mutate(chr=str_split_fixed(ex,pat="_",n=4)[,1],strand=str_split_fixed(ex,pat="_",n=4)[,4],
       ss1.E=ifelse(strand=="+",as.integer(str_split_fixed(E,pat="_",n=2)[,1]),ss1.junc),
       ss4.E=ifelse(strand=="+",ss4.junc,as.integer(str_split_fixed(E,pat="_",n=2)[,2]))) %>% 
       dplyr::select(-E) %>% 
left_join(.,ex,by=join_by(ss1.E==end,chr==seqid,strand==strand)) %>% 
dplyr::select(ex,ss1.junc,ss4.junc,ss1.E,ss4.E,start) %>% unique %>% 
mutate(len_1=ss1.E-start) %>% dplyr::select(-start) %>% 
group_by(ex,ss1.E,ss4.E,ss1.junc,ss4.junc) %>% 
summarize(len_1=min(len_1),.groups="drop")

cte_junc_surex <- cte_junc_filt %>% 
dplyr::select(-c(tr_id,stop_cod,reads,within_ex,UTR_len,ss1.ate,ss4.ate)) %>% unique %>% 
mutate(chr=str_split_fixed(ex,pat="_",n=4)[,1],strand=str_split_fixed(ex,pat="_",n=4)[,4],
       ss1.E=ifelse(strand=="+",as.integer(str_split_fixed(E,pat="_",n=2)[,1]),ss1.junc),
       ss4.E=ifelse(strand=="+",ss4.junc,as.integer(str_split_fixed(E,pat="_",n=2)[,2]))) %>% 
       dplyr::select(-E) %>% 
left_join(.,ex,by=join_by(ss4.E==start,chr==seqid,strand==strand)) %>% 
dplyr::select(ex,ss1.junc,ss4.junc,ss1.E,ss4.E,end) %>% unique %>% 
mutate(len_4=end-ss4.E) %>% dplyr::select(-end) %>% 
group_by(ex,ss1.E,ss4.E,ss1.junc,ss4.junc) %>% 
summarize(len_4=min(len_4),.groups="drop") %>% 
full_join(cte_junc_surex)

Warning message:
“There were 2 warnings in `mutate()`.
The first warning was:
ℹ In argument: `ss1.E = ifelse(...)`.
Caused by warning in `ifelse()`:
! NAs introduced by coercion
ℹ Run `dplyr::last_dplyr_warnings()` to see the 1 remaining warning.”
Warning message in left_join(., ex, by = join_by(ss1.E == end, chr == seqid, strand == :
“Detected an unexpected many-to-many relationship between `x` and `y`.
ℹ Row 1 of `x` matches multiple rows in `y`.
ℹ Row 105073 of `y` matches multiple rows in `x`.
ℹ If a many-to-many relationship is expected, set `relationship = "many-to-many"` to silence this warning.”
Warning message:
“There were 2 warnings in `mutate()`.
The first warning was:
ℹ In argument: `ss1.E = ifelse(...)`.
Caused by warning in `ifelse()`:
! NAs introduced by coercion
ℹ Run `dplyr::last_dplyr_warnings()` to see the 1 remaining warning.”
Warning message in left_join(., ex, by = join_by(ss4.E == start, chr == seqid, strand == :
“Detected an unexpected many-to-many relationship 

In [ ]:
cte_junc_surex %>% 
mutate(chr=str_split_fixed(ex,pat="_",n=4)[,1],strand=str_split_fixed(ex,pat="_",n=4)[,4],
       ss2=ifelse(strand=="+",str_split_fixed(ex,pat="_",n=4)[,2],ss4.junc),
       ss3=ifelse(strand=="+",ss1.junc,str_split_fixed(ex,pat="_",n=4)[,3]),
       ex=paste(chr,ss2,ss3,strand,ss1.E,ss4.E,sep="_")) %>% 
dplyr::select(-c(chr,strand,ss3,ss2,ss1.junc,ss4.junc)) %>% 
rowwise %>% mutate(across(starts_with("len_"),~min(.x,150))) %>% ungroup %>% 
mutate(e1_s=ss1.E-len_1,e2_e=ss4.E+len_4) %>% dplyr::rename(e1_e='ss1.E',e2_s='ss4.E') %>% 
dplyr::select(-starts_with("len")) %>% unique %>% 

pivot_longer(c(starts_with(c("e1","e2")))) %>% 
mutate(sur_ex=str_split_fixed(name,pat="_",n=2)[,1],
       ex_bord=str_split_fixed(name,pat="_",n=2)[,2],.keep="unused") %>% 
pivot_wider(id_cols=c(ex,sur_ex),values_from = value,names_from = "ex_bord") %>%  

mutate(chr=str_split_fixed(ex,pat="_",n=2)[,1],
       ex=paste(ex,sur_ex,sep="_"),
       score=0,
       strand=str_split_fixed(ex,pat="_",n=5)[,4],.before=everything()) %>% dplyr::select(-sur_ex) %>% 
relocate(c(s,e,ex),.after=chr) %>% arrange(chr,s,e) %>% 
write.table(.,file = "ATEPE_from_novel_junc_data/ATE_PE_e1_e2.PE_id.n260.bed",
          quote = F,row.names = F,col.names=F, sep="\t")

In [ ]:
cte_junc_surex %>% 
mutate(chr=str_split_fixed(ex,pat="_",n=4)[,1],strand=str_split_fixed(ex,pat="_",n=4)[,4],
       ss2=ifelse(strand=="+",str_split_fixed(ex,pat="_",n=4)[,2],ss4.junc),
       ss3=ifelse(strand=="+",ss1.junc,str_split_fixed(ex,pat="_",n=4)[,3]),
       ex=paste(chr,ss2,ss3,strand,ss1.E,ss4.E,sep="_")) %>% 
dplyr::select(-c(chr,strand,ss3,ss2,ss1.junc,ss4.junc)) %>% 
rowwise %>% mutate(across(starts_with("len_"),~min(.x,150))) %>% ungroup %>% 
mutate(e1_s=ss1.E-len_1,e2_e=ss4.E+len_4) %>% dplyr::rename(e1_e='ss1.E',e2_s='ss4.E') %>% 
dplyr::select(-starts_with("len")) %>% unique %>% 

pivot_longer(c(starts_with(c("e1","e2")))) %>% 
mutate(sur_ex=str_split_fixed(name,pat="_",n=2)[,1],
       ex_bord=str_split_fixed(name,pat="_",n=2)[,2],.keep="unused") %>% 
pivot_wider(id_cols=c(ex,sur_ex),values_from = value,names_from = "ex_bord") %>%  

mutate(chr=str_split_fixed(ex,pat="_",n=2)[,1],
       name=".",
       score=0,
       strand=str_split_fixed(ex,pat="_",n=5)[,4],.before=everything()) %>% dplyr::select(-sur_ex,-ex) %>% 
relocate(c(s,e,name),.after=chr) %>% arrange(chr,s,e) %>% unique %>% 
write.table(.,file = "ATEPE_from_novel_junc_data/ATE_PE_e1_e2.PE_id.uniq_sur_ex.n260.bed",
         quote = F,row.names = F,col.names=F, sep="\t")